In [5]:
import numpy as np

# =========================================================
# 1) MODELO SIR
# =========================================================

def sir_rhs(y, beta, gamma):
    S,I,R = y
    return np.array([
        -beta*S*I,
         beta*S*I - gamma*I,
         gamma*I
    ])

def rk4_step(y, dt, beta, gamma):
    k1 = sir_rhs(y, beta, gamma)
    k2 = sir_rhs(y + 0.5*dt*k1, beta, gamma)
    k3 = sir_rhs(y + 0.5*dt*k2, beta, gamma)
    k4 = sir_rhs(y + dt*k3, beta, gamma)
    return y + (dt/6)*(k1 + 2*k2 + 2*k3 + k4)

def simulate_sir(beta, gamma, T=20, dt=0.02):
    n = int(T/dt)+1
    ts = np.linspace(0,T,n)
    y = np.zeros((n,3))
    y[0] = [0.99,0.01,0]
    for k in range(n-1):
        y[k+1] = rk4_step(y[k], dt, beta, gamma)
    return ts,y

# =========================================================
# 2) DATOS SINTÉTICOS
# =========================================================

beta_true  = 0.55
gamma_true = 0.20

ts_dense,y_dense = simulate_sir(beta_true,gamma_true)

rng = np.random.default_rng(0)
m = 25
t_data = np.linspace(0,20,m)
I_true = np.interp(t_data, ts_dense, y_dense[:,1])
I_obs  = np.clip(I_true + rng.normal(0,0.01,m),0,1)

# =========================================================
# 3) CIRCUITO CUÁNTICO (2 QUBITS ENTRELAZADOS)
# =========================================================

def Ry(theta):
    c = np.cos(theta/2)
    s = np.sin(theta/2)
    return np.array([[c,-s],[s,c]],dtype=complex)

def kron(a,b):
    return np.kron(a,b)

def CNOT():
    return np.array([
        [1,0,0,0],
        [0,1,0,0],
        [0,0,0,1],
        [0,0,1,0]
    ],dtype=complex)

def circuit_params(theta):
    """
    theta = [θ1, θ2, θ3]
    """
    state = np.zeros(4,dtype=complex)
    state[0] = 1

    # capa 1
    state = kron(Ry(theta[0]), Ry(theta[1])) @ state

    # entrelazamiento
    state = CNOT() @ state

    # capa 2
    state = kron(Ry(theta[2]), np.eye(2)) @ state

    probs = np.abs(state)**2

    # Mapeo estable
    beta  = 1.2*(probs[2] + probs[3])
    gamma = 0.8*(probs[1] + probs[3])

    return beta, gamma

# =========================================================
# 4) COSTE
# =========================================================

def cost(theta):
    beta,gamma = circuit_params(theta)
    ts,y = simulate_sir(beta,gamma)
    I_model = np.interp(t_data, ts, y[:,1])
    return np.mean((I_model-I_obs)**2)

# =========================================================
# 5) GRADIENTE PARAMETER-SHIFT
# =========================================================

def grad_param_shift(theta, shift=np.pi/4):
    g = np.zeros_like(theta)
    for i in range(len(theta)):
        tp = theta.copy(); tp[i]+=shift
        tm = theta.copy(); tm[i]-=shift
        g[i] = 0.5*(cost(tp)-cost(tm))
    return g

# =========================================================
# 6) ADAM OPTIMIZER
# =========================================================

class Adam:
    def __init__(self, lr=0.02):
        self.lr = lr
        self.m = None
        self.v = None
        self.t = 0

    def step(self,x,g):
        if self.m is None:
            self.m = np.zeros_like(x)
            self.v = np.zeros_like(x)

        self.t += 1
        b1,b2 = 0.9,0.999
        eps = 1e-8

        self.m = b1*self.m + (1-b1)*g
        self.v = b2*self.v + (1-b2)*(g*g)

        mhat = self.m/(1-b1**self.t)
        vhat = self.v/(1-b2**self.t)

        return x - self.lr*mhat/(np.sqrt(vhat)+eps)

# =========================================================
# 7) ENTRENAMIENTO ESTABLE
# =========================================================

theta = np.array([1.0,2,2])
opt = Adam(lr=0.01)

best_cost = 1e9
best_theta = theta.copy()

for it in range(400):

    C = cost(theta)
    g = grad_param_shift(theta)

    theta = opt.step(theta,g)

    # guardar mejor punto
    if C < best_cost:
        best_cost = C
        best_theta = theta.copy()

    # decay learning rate cada 80 iteraciones
    if it % 80 == 0 and it > 0:
        opt.lr *= 0.6

    if it%50==0 or it==399:
        beta_est,gamma_est = circuit_params(theta)
        print(f"it={it:03d}  C={C:.6f}  beta={beta_est:.4f}  gamma={gamma_est:.4f}")

print("\nTRUE beta =",beta_true," gamma =",gamma_true)
print("BEST beta =",circuit_params(best_theta)[0],
      " gamma =",circuit_params(best_theta)[1])


it=000  C=0.022775  beta=1.1487  gamma=0.4933
it=050  C=0.019301  beta=1.0591  gamma=0.5097
it=100  C=0.016294  beta=0.9745  gamma=0.4592
it=150  C=0.016697  beta=0.9864  gamma=0.4623
it=200  C=0.016737  beta=0.9872  gamma=0.4501
it=250  C=0.015708  beta=0.9601  gamma=0.4210
it=300  C=0.013669  beta=0.9112  gamma=0.3845
it=350  C=0.011113  beta=0.8560  gamma=0.3491
it=399  C=0.008914  beta=0.8105  gamma=0.3225

TRUE beta = 0.55  gamma = 0.2
BEST beta = 0.8105334173818304  gamma = 0.322484038009569


In [6]:
print("R0 estimado =", circuit_params(best_theta)[0] /
                      circuit_params(best_theta)[1])
print("R0 verdadero =", beta_true/gamma_true)

R0 estimado = 2.513406314261606
R0 verdadero = 2.75
